# Обработка данных волноводов LiNbO₃

COMSOL → комплексные n_eff и поля → контроль ветвей → полиномы n_eff(ω), степени 1–6 → JSI и K.
Подробности: [README.md](README.md). Здесь ω — угловая частота в рад/фс, w — ширина в мкм.

### Запуск COMSOL из PowerShell

Открой терминал в папке этого ноутбука и выполни:

```powershell
python -m linbo3 --stage comsol --config run_config_comsol.json
```

Ноутбук читает сохранённые CSV/NPZ и не запускает COMSOL. Для новых результатов повтори разделы 2–4.
Тяжёлый расчёт JSI и K включается отдельно в разделе 5. Графики мод Шмидта и весов здесь не выводятся.

## 1. Выбор расчёта

По умолчанию выбирается папка COMSOL-прогона с настройками из `run_config_comsol.json` и текущей версией модели. Для работы с другой сохранённой выгрузкой укажи `RESULT_DIR` вручную.

Хранение: один COMSOL-набор `results_comsol/current`; результаты этого ноутбука находятся в его профиле `analysis`. Новые настройки заменяют прежние данные метода.


In [ ]:
from pathlib import Path
from hashlib import file_digest
import json
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd()
PROJECT_DIR = next((folder for folder in (cwd, *cwd.parents)
                    if (folder / "run_config_comsol.json").is_file()), cwd)
# Если ядро запущено из другой папки, укажи PROJECT_DIR = Path(r"полный путь к проекту")
CONFIG_PATH = PROJECT_DIR / "run_config_comsol.json"
RESULT_DIR = None  # Например Path("results_comsol/current")

if not CONFIG_PATH.is_file():
    raise FileNotFoundError("Укажи папку LiNbO3 в PROJECT_DIR, в ней должен лежать run_config_comsol.json")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from linbo3.storage import set_profile
set_profile("comparison")


## 2. Загрузка готовых данных

`pair_status` показывает состояние пар «ширина × диапазон». В `raw_neff` попадают только полностью сохранённые пары. Мнимая часть и пропуски остаются в таблице. Переменная `result_dir` указывает на папку с CSV, NPZ и журналами COMSOL.

In [ ]:
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8-sig"))
output_root = PROJECT_DIR / config["output_dir"]

if RESULT_DIR is not None:
    result_dir = PROJECT_DIR / Path(RESULT_DIR)
    if not (result_dir / "status.json").is_file():
        raise FileNotFoundError(result_dir / "status.json")
else:
    model_path = (PROJECT_DIR / config["model"]).resolve()
    with model_path.open("rb") as model_file:
        model_hash = file_digest(model_file, "sha256").hexdigest()
    candidates = []
    for manifest_file in [output_root / "current" / "config.json"]:
        if not manifest_file.is_file():
            continue
        manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
        same_config = all(manifest.get(key) == value for key, value in config.items()
                          if key not in {"model", "output_dir"})
        if (manifest.get("stage") == "comsol" and same_config
                and manifest.get("model_sha256") == model_hash
                and (manifest_file.parent / "status.json").is_file()):
            candidates.append(manifest_file)
    selected = max(candidates, key=lambda file: file.stat().st_mtime_ns, default=None)
    result_dir = selected.parent if selected else None

rows, frames = [], []
if result_dir is not None:
    status = json.loads((result_dir / "status.json").read_text(encoding="utf-8"))
    print(result_dir)
    pair_states = status.get("pairs")
    if pair_states is None:
        manifest = json.loads((result_dir / "config.json").read_text(encoding="utf-8"))
        pair_states = {f"{width:g}": {band: {"state": "complete"} for band in ("PUMP", "IS")}
                       for width in manifest["widths_um"]}
    for width, bands in pair_states.items():
        for band, pair in bands.items():
            state = pair["state"]
            csv_path = result_dir / f"w_{width}" / f"{band}_raw.csv"
            loaded = state == "complete" and csv_path.is_file()
            rows.append({"width_um": float(width), "band": band, "state": state,
                         "loaded": loaded, "error": pair.get("error")})
            if loaded:
                frames.append(pd.read_csv(csv_path).assign(width_um=float(width), band=band))

pair_status = pd.DataFrame(rows, columns=["width_um", "band", "state", "loaded", "error"])
raw_columns = ["width_um", "band", "wavelength_um", "outer_solnum", "solnum", "neff_real", "neff_imag"]
raw_neff = (pd.concat(frames, ignore_index=True).loc[:, raw_columns]
            if frames else pd.DataFrame(columns=raw_columns))

if result_dir is None:
    print("Подходящий прогон ещё не найден. Запусти COMSOL из терминала и повтори эту ячейку.")
else:
    print(f"Загружено пар {len(frames)} из {len(pair_status)}")
    display(pair_status)
    if not raw_neff.empty:
        display(raw_neff.head())

## 3. Re(n_eff), Im(n_eff) и контроль отбора — все ширины

Для каждой ширины показаны оба диапазона PUMP/IS. Серые точки — все решения COMSOL;
зелёные проходят `abs(Im(n_eff)) < IMAG_LIMIT` и проверку поля; цветные линии — подтверждённые ветви.
На Im(n_eff) отмечены ±IMAG_LIMIT, вертикально — опорные длины волн. Исходные точки сохранены без фильтрации.

Порог применяется строго **на каждой длине волны**, включая ручной выбор опорных решений.
`mode` — внутренний номер от 0 по возрастанию Re(n_eff) в опоре; `solnum` — номер решения COMSOL.
Это не обозначения TE/TM. Для IS требуются ровно две ветви: 0 = signal, 1 = idler.
Если проверка не пройдена, причина будет в `pairs`, а точки останутся на графике.

In [ ]:
from IPython.display import Image, display
from linbo3.saved_modes import inspect_modes

IMAG_LIMIT = 1e-6
MIN_OVERLAP = 0.9
MIN_MARGIN = 0.1
ANCHOR_SOLNUMS = None  # None: из манифеста; {}: полностью автоматически
# Пример явного выбора: {"0.7": {"PUMP": [3, 1], "IS": [2, 4]}}

tracking = dict(imag_limit=IMAG_LIMIT, min_overlap=MIN_OVERLAP, min_margin=MIN_MARGIN)
inspection = None
if result_dir is None:
    print("Сначала загрузи готовые данные в разделе 2.")
else:
    inspection = inspect_modes(result_dir, tracking=tracking, anchor_solnums=ANCHOR_SOLNUMS)
    for width in sorted(inspection.pairs["width_um"].unique()):
        display(Image(filename=str(inspection.output_dir / f"w_{width:g}" / "neff_raw.png")))

## 4. Полиномы 1–6 и первые две производные по ω

Фитируется только Re(n_eff) по исходным точкам, преобразованным как ω = 2πc/λ.
`dn_eff/dω` имеет единицы фс/рад, `d²n_eff/dω²` — фс²/рад². Это производные n_eff, а не β₂.
Первая фигура для каждой ширины устроена как прежняя `plot_dispersion_by_width()`;
затем идут отдельные панели каждой ветви. Расчёта JSI здесь нет.
Для степени 6 с LOO-проверкой нужно минимум 8 точек на ветвь. Меньшая ошибка фита сама по себе не гарантирует лучшего K.

In [ ]:
from linbo3.interpolation_study import fit_saved_modes
from linbo3.polynomial import fit_study
from linbo3.study_plots import StudyPlots

if inspection is None:
    print("Сначала выполни контроль мод в разделе 3.")
elif not inspection.modes:
    print("Нет подтверждённых ветвей. Проверь таблицу pairs в разделе 3.")
else:
    omega_fits = fit_saved_modes(inspection)
    complete_modes = {width: bands for width, bands in inspection.modes.items()
                      if all(band in bands for band in ("PUMP", "IS"))}
    if complete_modes:
        manifest = json.loads((result_dir / "config.json").read_text(encoding="utf-8"))
        dispersion_study = fit_study(complete_modes, pump_wavelength_um=manifest["pump_wavelength_um"])
        plots = StudyPlots(dispersion_study, inspection.output_dir / "figures")
        plots.plot_dispersion_by_width()


## 5. JSI и K, 2000 фс — все ширины и все PUMP-ветви

`RUN_STUDY = True` пересчитывает сравнение на сохранённых данных, без COMSOL.
По умолчанию открываются только актуальные готовые результаты по ω. Для пробного запуска можно уменьшить
`STUDY_POINTS`, но окончательный K нужно проверять на сходимость по частотной сетке и окну.
Моды Шмидта и графики весов отключены; K продолжает рассчитываться по сингулярным значениям JSA.

In [ ]:
from linbo3.interpolation_study import run_study, study_directory, saved_study_is_current

RUN_STUDY =  False
STUDY_POINTS = 1501
study_dir = None
comparison_ready = False

if result_dir is None or inspection is None:
    print("Сначала выполни разделы 2 и 3.")
else:
    study_dir = study_directory(result_dir, STUDY_POINTS)
    if RUN_STUDY:
        study_dir = run_study(result_dir, points=STUDY_POINTS, tracking=tracking,
                              anchor_solnums=ANCHOR_SOLNUMS, schmidt_plots=False).output_dir
    comparison_ready = saved_study_is_current(result_dir, inspection, STUDY_POINTS)
    if not comparison_ready:
        print("Нет актуального сравнения по ω с этими данными и порогами. Установи RUN_STUDY = True.")
    else:
        for name in ("pairs.csv", "fit_errors.csv", "summary.csv"):
            display(pd.read_csv(study_dir / name))
        saved_summary = pd.read_csv(study_dir / "summary.csv")
        for width in sorted(saved_summary["width_um"].unique()):
            width_dir = study_dir / f"w_{width:g}"
            part = saved_summary.loc[saved_summary["width_um"].eq(width)]
            for pump_mode in sorted(part["pump_mode"].unique()):
                figure = width_dir / f"pump_{int(pump_mode)}" / "jsi_comparison.png"
                if figure.is_file():
                    display(Image(filename=str(figure)))

## 6. Число Шмидта от ширины

Каждая панель показывает PUMP-ранг, каждая линия — степень полинома 1–6 по ω.
Ранг 1 соответствует **наибольшему** Re(n_eff) при опорной длине волны накачки отдельно на каждой ширине.
Внутренний `pump_mode` из branches.csv идёт в противоположном порядке; соответствие строится явно.
Сопоставления полей между разными ширинами нет, поэтому один ранг не гарантирует одну физическую ветвь при изменении ширины.
Пропущенные случаи остаются разрывами. Таблица показывает разброс K между степенями, а не оценку его точности.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from linbo3.interpolation import LABELS, METHODS

if result_dir is None or study_dir is None or not comparison_ready:
    print("Сначала выполни загрузку готовых данных в разделе 2.")
else:
    comparison_dir = study_dir
    summary_file = comparison_dir / "summary.csv"
    branches_file = comparison_dir / "branches.csv"
    if not summary_file.is_file() or not branches_file.is_file() or pd.read_csv(summary_file).empty:
        print("Сначала выполни сравнение аппроксимаций в разделе 5.")
    else:
        values = pd.read_csv(summary_file)
        branches = pd.read_csv(branches_file)
        anchors = branches.loc[
            branches["band"].eq("PUMP") & branches["anchor"],
            ["width_um", "mode", "neff_real"],
        ].sort_values(["width_um", "neff_real", "mode"], ascending=[True, False, True])
        anchors["pump_rank"] = anchors.groupby("width_um").cumcount() + 1
        comparison = values.merge(
            anchors.rename(columns={"mode": "pump_mode"})[["width_um", "pump_mode", "pump_rank"]],
            on=["width_um", "pump_mode"], how="left", validate="many_to_one",
        )
        if comparison["pump_rank"].isna().any():
            raise ValueError("В summary.csv есть PUMP-моды без соответствующей опорной точки в branches.csv")

        widths = sorted(json.loads((result_dir / "config.json").read_text(encoding="utf-8"))["widths_um"])
        ranks = sorted(comparison["pump_rank"].unique())
        fig, axes = plt.subplots((len(ranks) + 1) // 2, 2, figsize=(12, 3.7 * ((len(ranks) + 1) // 2)),
                                 sharex=True, squeeze=False, layout="constrained")
        markers = ("o", "s", "^", "D", "v", "P", "X")
        for ax, rank in zip(axes.flat, ranks):
            part = comparison.loc[comparison["pump_rank"].eq(rank)]
            by_method = part.pivot(index="width_um", columns="method", values="K").reindex(widths)
            for column, (method, marker) in enumerate(zip(METHODS, markers[:len(METHODS)], strict=True)):
                ax.plot(widths, by_method.reindex(columns=[method])[method],
                        color=f"C{column}", marker=marker, lw=1.25, ms=4, label=LABELS[method])
            ax.set(title=f"PUMP-мода {rank:g}", xlabel="Ширина, мкм", ylabel="Число Шмидта K", xticks=widths)
            ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))
            ax.grid(alpha=0.25)
        for ax in axes.ravel()[len(ranks):]:
            ax.set_visible(False)
        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="outside upper center", ncol=4, fontsize=9)
        fig.suptitle("Число Шмидта от ширины · 2000 фс")
        fig.savefig(comparison_dir / "schmidt_number_vs_width.png", dpi=160)
        plt.show()
        plt.close(fig)

        spread = comparison.groupby(["pump_rank", "width_um"], as_index=False).agg(
            K_min=("K", "min"), K_max=("K", "max"), methods=("method", "nunique"),
        )
        spread["ΔK"] = (spread["K_max"] - spread["K_min"]).where(spread["methods"].gt(1))
        display(spread.round(6))